# LFI — Inference Methods Demo

This notebook demonstrates four inference methods provided by the `lfi` package, all on the same toy problem:
a 2-D Gaussian simulator with a uniform prior.

| Script | Class | Backend |
|---|---|---|
| `native.py` | `ABCRejection` | pure NumPy |
| `native.py` | `SMCInference` | pure NumPy |
| `r2omc.py` | `R2OMC` | JAX |
| `sbi.py` | `NPECSingleRound` | PyTorch / sbi |
| `elfi.py` | `RejectionSampling` | ELFI |

## Setup

In [ ]:
import numpy as np
import lfi

prior      = lfi.priors.UniformPrior(dim=2, low=-3, high=3)
simulator  = lfi.simulators.GaussianNoise(dim=2, dim_y=2, sigma_noise=0.1)
obs        = np.array([[1.5, 1.5]])

BUDGET      = 1_000
NOF_SAMPLES = 100

print(f"Prior:      {prior}")
print(f"Simulator:  {simulator}")
print(f"Observation: {obs}")

---
## 1 · ABC Rejection  (`native.ABCRejection`)

Samples `budget` parameter vectors from the prior, simulates each one, then keeps the
top `quantile` fraction whose output is closest to the observation.

In [ ]:
abc = lfi.inference.native.ABCRejection(
    prior=prior, simulator=simulator, observation=obs
)
samples_abc = abc.fit_and_sample(
    budget=BUDGET,
    nof_samples=NOF_SAMPLES,
    fit_kwargs={"quantile": 0.1},
)
print(f"samples shape: {samples_abc.shape}")

In [ ]:
abc.plot_posterior_samples(limits=(-3, 3), show=True)

---
## 2 · SMC Inference  (`native.SMCInference`)

Sequential Monte Carlo ABC: runs several rounds with a decreasing tolerance sequence,
progressively focusing the particle cloud around the observation.

In [ ]:
smc = lfi.inference.native.SMCInference(
    prior=prior, simulator=simulator, observation=obs
)
samples_smc = smc.fit_and_sample(
    budget=BUDGET,
    nof_samples=NOF_SAMPLES,
    fit_kwargs={"tolerance_sequence": [1.0, 0.5, 0.25, 0.1]},
)
print(f"samples shape: {samples_smc.shape}")

In [ ]:
smc.plot_posterior_samples(limits=(-3, 3), show=True)

---
## 3 · R2OMC  (`r2omc.R2OMC`)

Gradient-based likelihood-free inference using JAX.  Optimises seeds in the
common-random-numbers space and builds bounding boxes around the solutions to
importance-sample the posterior.

In [ ]:
r2omc = lfi.inference.r2omc.R2OMC(
    prior=prior, simulator=simulator, observation=obs
)
samples_r2omc = r2omc.fit_and_sample(budget=BUDGET, nof_samples=NOF_SAMPLES)
print(f"samples shape: {samples_r2omc.shape}")

In [ ]:
r2omc.plot_posterior_samples(limits=(-3, 3), show=True)

---
## 4 · NPE-C  (`sbi.NPECSingleRound`)

Automatic Posterior Transformation (APT / NPE-C): trains a normalising flow to
directly approximate the posterior.  Requires `lfi[torch-cpu]` or `lfi[torch-gpu]`.

In [ ]:
try:
    npec = lfi.inference.sbi.NPECSingleRound(
        prior=prior, simulator=simulator, observation=obs
    )
    samples_npec = npec.fit_and_sample(budget=BUDGET, nof_samples=NOF_SAMPLES)
    print(f"samples shape: {samples_npec.shape}")
except ImportError as e:
    print(f"Skipped (missing dependency): {e}")

In [ ]:
try:
    npec.plot_posterior_samples(limits=(-3, 3), show=True)
except NameError:
    print("NPE-C was skipped above.")

---
## 5 · ELFI Rejection Sampling  (`elfi.RejectionSampling`)

Rejection ABC implemented via the ELFI framework.  Requires `lfi[elfi]`.

In [ ]:
try:
    elfi_rs = lfi.inference.elfi.RejectionSampling(
        prior=prior, simulator=simulator, observation=obs
    )
    samples_elfi = elfi_rs.fit_and_sample(budget=BUDGET, nof_samples=NOF_SAMPLES)
    print(f"samples shape: {samples_elfi.shape}")
except ImportError as e:
    print(f"Skipped (missing dependency): {e}")

In [ ]:
try:
    elfi_rs.plot_posterior_samples(limits=(-3, 3), show=True)
except NameError:
    print("ELFI was skipped above.")